In [ ]:
import torch
import torch.nn as nn
import numpy as np
import itertools
import networkx as nx
import matplotlib.pyplot as plt

# Re-implementing the dataset generation so the notebook is self-contained
def make_circles(n_samples=300, noise=0.05, seed=42, center=(0.0, 0.0)):
    np.random.seed(seed)
    angles = np.random.uniform(0, 2*np.pi, n_samples)
    radii = np.where(np.random.rand(n_samples) > 0.5, 1.0, 3.0) + np.random.randn(n_samples) * noise
    X = np.c_[radii * np.cos(angles), radii * np.sin(angles)]
    X += np.array(center)
    y = (radii > 2.0).astype(int)
    return X, y

def make_xor(n_samples=300, noise=0.0, seed=42, center=(0.0, 0.0)):
    np.random.seed(seed)
    X = np.random.randn(n_samples, 2)
    y = np.logical_xor(X[:, 0] > 0, X[:, 1] > 0).astype(int)
    X += np.array(center)
    return X, y

def make_linearly_separable(n_samples=300, noise=0.1, seed=42, center=(0.0, 0.0)):
    np.random.seed(seed)
    X = np.random.randn(n_samples, 2)
    y = (X[:, 0] + X[:, 1] > 0).astype(int)
    X += np.array(center)
    return X, y



In [ ]:


class PyTorchDynamicNetwork(nn.Module):
    """
    Adjacency-matrix dynamic network with hard-freeze pathway separation.

    Architecture
    ────────────
    • W[max×max]          : weight matrix (single nn.Parameter)
    • M[max×max]          : structural mask (which connections exist)
    • trainable_mask[max×max] : gradient mask (which connections can learn)

    The gradient hook ensures:
        W.grad *= trainable_mask
    so frozen weights receive EXACTLY ZERO gradient — no optimizer state update,
    no momentum, nothing.  They are physically frozen.

    Lifecycle
    ─────────
    1. Network starts small (input + output + 4 hidden).
       All existing connections have trainable_mask = 1 → fully plastic.

    2. Train on Task 1.  Network learns.  Fisher (EMA of g²) builds up
       for weights that are actively used.

    3. When a new distribution arrives (detected by rising EWC stress):
       a. autonomous_freeze() sets trainable_mask = 0 for ALL current connections
          that have significant Fisher importance.
       b. grow_neurons() adds new neurons with trainable_mask = 1.
       c. New neurons connect to inputs AND to the output → they form a
          parallel pathway that can learn the new task without any gradient
          flowing through old frozen weights.

    4. Replay ensures the new pathway doesn't accidentally counteract the
       old pathway's contribution to the output.

    Why this works where EWC/SI/AGEM failed
    ────────────────────────────────────────
    Those methods tried to DISCOURAGE the optimizer from changing old weights
    (via penalties or gradient scaling).  But in a recurrent forward pass,
    the optimizer always found paths through unprotected weights.

    Hard-freeze doesn't discourage — it PREVENTS.  Zero gradient = zero change.
    The optimizer has no choice but to use the new neurons.
    """

    def __init__(self, input_dim: int, output_dim: int,
                 max_neurons: int = 200, steps: int = 3):
        super().__init__()
        self.input_dim   = input_dim
        self.output_dim  = output_dim
        self.max_neurons = max_neurons
        self.steps       = steps

        self.active_neurons = input_dim + output_dim + 4

        # ── Learnable parameters ──────────────────────────────────────────
        self.W = nn.Parameter(torch.randn(max_neurons, max_neurons) * 0.1)
        self.b = nn.Parameter(torch.zeros(max_neurons))

        # ── Structural mask (which connections exist) ─────────────────────
        self.register_buffer('M', torch.zeros(max_neurons, max_neurons))

        # ── Gradient mask (which connections can learn) ───────────────────
        # 1.0 = trainable,  0.0 = frozen (hard zero gradient)
        self.register_buffer('trainable_mask',
                             torch.ones(max_neurons, max_neurons))

        # ── Bias freeze mask ─────────────────────────────────────────────
        self.register_buffer('bias_trainable',
                             torch.ones(max_neurons))

        # ── Per-weight gradient conflict statistics ───────────────────────────
        #
        # We explicitly measure if the CURRENT task is fighting REPLAY memory.
        #
        # stress_num: EMA of (-g_task * g_replay).
        #             Positive if they pull in opposite directions.
        #
        # stress_den: EMA of (|g_task| * |g_replay|).
        #             Normalization factor.
        #
        # stress = stress_num / (stress_den + 1e-8)
        #
        #   stress ≈ -1.0  →  Consistent agreement (single task learning)
        #   stress ≈ 0.0   →  Independent noise (converged at minimum)
        #   stress > 0.0   →  True conflict (new task destroying old knowledge)
        #
        self.register_buffer('stress_num', torch.zeros(max_neurons, max_neurons))
        self.register_buffer('stress_den', torch.zeros(max_neurons, max_neurons))

        # ── Gradient hooks (registered once) ─────────────────────────────
        self._hook_registered = False

        self._init_random_connections()

    # ─────────────────────────────────────────────────────────────────────
    def _init_random_connections(self):
        for i in range(self.active_neurons):
            for j in range(self.active_neurons):
                if i != j and i >= self.input_dim:
                    if torch.rand(1).item() > 0.5:
                        self.M[i, j] = 1.0

    # ─────────────────────────────────────────────────────────────────────
    def _register_hooks(self):
        """Register gradient hooks that enforce the trainable_mask."""
        if self._hook_registered:
            return

        def _w_hook(grad):
            # Hard-zero gradients for frozen connections
            return grad * self.trainable_mask * self.M

        def _b_hook(grad):
            return grad * self.bias_trainable

        self.W.register_hook(_w_hook)
        self.b.register_hook(_b_hook)
        self._hook_registered = True

    # ─────────────────────────────────────────────────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Ensure hooks are registered on first forward
        if not self._hook_registered:
            self._register_hooks()

        batch = x.size(0)
        state = torch.zeros(batch, self.max_neurons, device=x.device)
        state[:, :self.input_dim] = x

        W_eff = self.W * self.M

        for _ in range(self.steps):
            new_state = torch.relu(torch.matmul(state, W_eff.T) + self.b)
            state = torch.cat([x, new_state[:, self.input_dim:]], dim=1)

        return state[:, self.input_dim: self.input_dim + self.output_dim]

    # ─────────────────────────────────────────────────────────────────────
    def update_fisher(self, beta: float = 0.999):
        """
        Update Fisher (EMA of grad²) as a diagnostic signal.
        Called every batch AFTER backward().
        """
        with torch.no_grad():
            if self.W.grad is not None:
                grad_sq = (self.W.grad ** 2) * self.M
                self.fisher.mul_(beta).add_((1 - beta) * grad_sq)

    # ─────────────────────────────────────────────────────────────────────
    def update_stress(self, g_task: torch.Tensor, g_replay: torch.Tensor, beta: float = 0.99):
        """
        Update explicit task-vs-replay conflict statistics.
        
        Args:
            g_task: Gradient of W computed on the current task's batch.
            g_replay: Gradient of W computed on a batch from the replay buffer.
        """
        with torch.no_grad():
            # Only consider active connections that are trainable
            active = self.M * self.trainable_mask
            g_t = (g_task * active).detach()
            g_r = (g_replay * active).detach()
            
            # The product is negative if they point in opposite directions.
            # We negate it so that conflict is POSITIVE.
            conflict = -(g_t * g_r)
            magnitude = g_t.abs() * g_r.abs()
            
            self.stress_num.mul_(beta).add_((1 - beta) * conflict)
            self.stress_den.mul_(beta).add_((1 - beta) * magnitude)

    # ─────────────────────────────────────────────────────────────────────
    def get_stress_matrix(self) -> torch.Tensor:
        """
        Per-weight conflict fraction in [-1.0, 1.0].
        
        Only active trainable connections have non-zero stress.
        """
        with torch.no_grad():
            stress = self.stress_num / (self.stress_den + 1e-8)
            # Clip safely
            stress = stress.clamp(min=-1.0, max=1.0)
            return stress * self.M * self.trainable_mask

    # ─────────────────────────────────────────────────────────────────────
    def get_network_stress(self, conflict_threshold: float = 0.3) -> float:
        """
        Fraction of active trainable connections that are in high conflict.
        
        Returns a value in [0, 1]:
          0.0  →  No connections are conflicted (stable or just noise)
          ~1.0 →  All trainable connections are conflicted
        """
        with torch.no_grad():
            stress = self.get_stress_matrix()
            active_trainable = (self.M * self.trainable_mask).bool()
            if active_trainable.sum() == 0:
                return 0.0

            high_conflict = (stress[active_trainable] > conflict_threshold).float()
            return high_conflict.mean().item()

    # ─────────────────────────────────────────────────────────────────────
    def stress_freeze(self, threshold: float = 0.3) -> int:
        """
        Autonomously freeze the most-conflicted trainable connections.
        
        Args:
            threshold: Any trainable connection with a stress > threshold
                       is permanently frozen.
        """
        with torch.no_grad():
            stress = self.get_stress_matrix()
            active_trainable = (self.M * self.trainable_mask).bool()
            if active_trainable.sum() == 0:
                return 0

            freeze_mask = (stress >= threshold) & active_trainable
            n_frozen = int(freeze_mask.sum().item())

            self.trainable_mask[freeze_mask] = 0.0

            # Also freeze biases of neurons whose incoming connections
            # are now mostly frozen (they are 'old' neurons)
            for i in range(self.input_dim, self.active_neurons):
                incoming = self.M[i, :self.active_neurons]
                if incoming.sum() == 0:
                    continue
                frozen_frac = (
                    (1 - self.trainable_mask[i, :self.active_neurons])
                    [incoming.bool()].mean()
                )
                if frozen_frac > 0.5:
                    self.bias_trainable[i] = 0.0

            return n_frozen

    # ─────────────────────────────────────────────────────────────────────
    def get_ewc_stress(self) -> float:
        """
        Compute how much "stress" the network is under.

        Stress = mean Fisher of currently-trainable connections.
        High stress = existing trainable weights are receiving large gradients
        = the current task is trying hard to change them = potential conflict.
        """
        active_trainable = self.trainable_mask * self.M
        if active_trainable.sum() == 0:
            return 0.0
        return (self.fisher * active_trainable).sum().item() / active_trainable.sum().item()

    # ─────────────────────────────────────────────────────────────────────
    def autonomous_freeze(self, fisher_threshold_percentile: float = 50.0):
        """
        AUTONOMOUSLY freeze connections that have high Fisher importance.

        This is NOT manual task-boundary freezing.  It is triggered by the
        training loop when stress exceeds a threshold.

        Process:
        1. Look at Fisher values for currently-trainable connections.
        2. Connections with Fisher above the percentile threshold → freeze
           (trainable_mask = 0).
        3. Connections below threshold → stay trainable (they weren't
           important, so they can be reused).

        After freezing:
        - The network CANNOT modify these connections anymore (gradient = 0).
        - New neurons must be grown to provide fresh pathway capacity.
        """
        with torch.no_grad():
            # Only consider currently-trainable + active connections
            active = (self.trainable_mask * self.M).bool()
            if active.sum() == 0:
                return 0

            fisher_vals = self.fisher[active]
            if fisher_vals.numel() == 0:
                return 0

            threshold = torch.quantile(fisher_vals, fisher_threshold_percentile / 100.0)

            # Freeze connections with Fisher >= threshold
            freeze_mask = (self.fisher >= threshold) & active
            n_frozen = freeze_mask.sum().item()

            self.trainable_mask[freeze_mask] = 0.0

            # Also freeze biases of neurons whose incoming connections are
            # mostly frozen (they are "old" neurons now)
            for i in range(self.input_dim, self.active_neurons):
                incoming = self.M[i, :self.active_neurons]
                if incoming.sum() == 0:
                    continue
                frozen_frac = (1 - self.trainable_mask[i, :self.active_neurons])[incoming.bool()].mean()
                if frozen_frac > 0.5:
                    self.bias_trainable[i] = 0.0

            return n_frozen

    # ─────────────────────────────────────────────────────────────────────
    def grow_neuron(self, num_connections: int = 5):
        """
        Activate next pre-allocated neuron.

        New neuron has:
          • trainable_mask = 1 for all its connections → fully plastic
          • Fisher = 0 → not considered important (yet)
          • Small random weights → doesn't disrupt existing output
          • Connects to BOTH inputs and outputs → forms a parallel pathway

        The output neuron connections to the new neuron are TRAINABLE,
        while output connections to old neurons remain FROZEN.
        """
        if self.active_neurons >= self.max_neurons:
            print("  [network] Max capacity reached.")
            return

        new_idx = self.active_neurons
        self.active_neurons += 1

        with torch.no_grad():
            # Clear the new neuron's slot
            self.W.data[new_idx, :] = 0.0
            self.W.data[:, new_idx] = 0.0
            self.b.data[new_idx]    = 0.0

            # New neuron is fully trainable
            self.trainable_mask[new_idx, :] = 1.0
            self.trainable_mask[:, new_idx] = 1.0
            self.bias_trainable[new_idx]    = 1.0

            # Reset stress buffers for the new neuron
            self.stress_num[new_idx, :] = 0.0
            self.stress_num[:, new_idx] = 0.0
            self.stress_den[new_idx, :] = 0.0
            self.stress_den[:, new_idx] = 0.0

        # Wire incoming: from inputs + existing hidden neurons
        sources = [s for s in range(new_idx)
                   if s < self.input_dim or
                   s >= self.input_dim + self.output_dim]
        if sources:
            # Prefer connecting to input neurons (direct fresh signal)
            input_sources = [s for s in sources if s < self.input_dim]
            hidden_sources = [s for s in sources if s >= self.input_dim + self.output_dim]

            # Always connect to all inputs
            for s in input_sources:
                self.M[new_idx, s] = 1.0
                with torch.no_grad():
                    self.W.data[new_idx, s] = torch.randn(1).item() * 0.1

            # Connect to a few random hidden neurons
            if hidden_sources:
                n_hidden_conn = min(num_connections, len(hidden_sources))
                chosen = np.random.choice(hidden_sources, n_hidden_conn, replace=False)
                for s in chosen:
                    self.M[new_idx, s] = 1.0
                    with torch.no_grad():
                        self.W.data[new_idx, s] = torch.randn(1).item() * 0.05

        # Wire outgoing: to output neurons AND a few hidden neurons
        output_indices = list(range(self.input_dim, self.input_dim + self.output_dim))
        for out_idx in output_indices:
            self.M[out_idx, new_idx] = 1.0
            self.trainable_mask[out_idx, new_idx] = 1.0  # explicitly trainable
            with torch.no_grad():
                self.W.data[out_idx, new_idx] = 0.0  # start at 0 — no disruption

        # Also connect to a few other hidden neurons
        hidden_targets = [t for t in range(self.input_dim + self.output_dim, new_idx)]
        if hidden_targets:
            n_out = min(num_connections, len(hidden_targets))
            chosen = np.random.choice(hidden_targets, n_out, replace=False)
            for t in chosen:
                self.M[t, new_idx] = 1.0
                with torch.no_grad():
                    self.W.data[t, new_idx] = torch.randn(1).item() * 0.05

    # ─────────────────────────────────────────────────────────────────────
    def n_frozen(self) -> int:
        """Number of frozen connections."""
        active = self.M.bool()
        return int((active & ~self.trainable_mask.bool()).sum().item())

    def n_trainable(self) -> int:
        """Number of trainable connections."""
        return int((self.M * self.trainable_mask).sum().item())



In [ ]:



# ─────────────────────────────────────────────────────────────────────────────
class ReplayBuffer:
    """
    Flat reservoir-sampled episodic memory.

    Purpose: provide a stream of MIXED past+current data to the training loss.
    This is NOT used for forgetting detection — that job belongs to the
    network's own gradient stress signal.

    Reservoir sampling ensures every past sample has equal probability of
    being retained regardless of when it arrived.
    """

    def __init__(self, capacity: int = 500):
        self.capacity = capacity
        self.X: torch.Tensor | None = None
        self.y: torch.Tensor | None = None
        self.n_seen = 0

    def push(self, X: torch.Tensor, y: torch.Tensor):
        X, y = X.detach().cpu(), y.detach().cpu()
        for xi, yi in zip(X, y):
            xi, yi = xi.unsqueeze(0), yi.unsqueeze(0)
            if self.n_seen < self.capacity:
                self.X = xi if self.X is None else torch.cat([self.X, xi], 0)
                self.y = yi if self.y is None else torch.cat([self.y, yi], 0)
            else:
                j = np.random.randint(0, self.n_seen + 1)
                if j < self.capacity:
                    self.X[j] = xi[0]
                    self.y[j] = yi[0]
            self.n_seen += 1

    def sample(self, n: int, device):
        if self.X is None or len(self.X) == 0:
            return None, None
        n = min(n, len(self.X))
        idx = torch.randperm(len(self.X))[:n]
        return self.X[idx].to(device), self.y[idx].to(device)

    def __len__(self):
        return len(self.X) if self.X is not None else 0


# ─────────────────────────────────────────────────────────────────────────────
def _reset_adam_for_neuron(optimizer, net, ni: int):
    """Zero out Adam momentum/variance for a freshly grown neuron."""
    if net.W not in optimizer.state:
        return
    s_W = optimizer.state[net.W]
    for key in ('exp_avg', 'exp_avg_sq'):
        if key in s_W:
            s_W[key][ni, :] = 0.0
            s_W[key][:, ni] = 0.0
    if net.b in optimizer.state:
        s_b = optimizer.state[net.b]
        for key in ('exp_avg', 'exp_avg_sq'):
            if key in s_b:
                s_b[key][ni] = 0.0


# ─────────────────────────────────────────────────────────────────────────────
def train_one_task_gpu(
    net: PyTorchDynamicNetwork,
    X: torch.Tensor,
    y: torch.Tensor,
    epochs: int,
    lr: float,
    *,
    replay: ReplayBuffer = None,
    device='cpu'
):
    """
    Training loop driven by the network's own internal stress signal.

    ╔══════════════════════════════════════════════════════════════════╗
    ║  Core principle (no cheating, no task labels)                   ║
    ║                                                                  ║
    ║  Each weight tracks explicit gradient covariance:                ║
    ║    g_task   = gradient from current task's batch                ║
    ║    g_replay = gradient from replay memory batch                 ║
    ║                                                                  ║
    ║    stress   = EMA(-g_task * g_replay) / EMA(|g_task| * |g_replay|) 
    ║                                                                  ║
    ║  Interpretation:                                                 ║
    ║    stress < 0  →  Task and Replay agree (single task learning)  ║
    ║    stress ≈ 0  →  Independent noise (converged at minimum)      ║
    ║    stress > 0  →  True conflict (new task destroying old memory)║
    ║                                                                  ║
    ║  get_network_stress() returns the fraction of connections that  ║
    ║  have a stress > 0.3.                                           ║
    ║                                                                  ║
    ║  When conflict crosses the threshold → weights "feel" it and    ║
    ║  signal: "I can't hold both things — give me space."            ║
    ║    → stress_freeze(): lock the highly-conflicted connections     ║
    ║    → grow_neuron():   add fresh neurons for the new learning     ║
    ║                                                                  ║
    ║  Why this works                                                  ║
    ║    Task 1 only (no replay conflict):                             ║
    ║      Early: gradients point towards minimum. stress < 0.        ║
    ║      Late: gradients are noise. noise averages to 0. stress ≈ 0.║
    ║      → conflict never rises → no freeze fires.                  ║
    ║                                                                  ║
    ║    Task 2 starts (XOR fights Circles in replay):                 ║
    ║      Circles replay: pushes weight +                             ║
    ║      XOR forward:    pushes weight −                             ║
    ║      g_task * g_replay is consistently negative.                 ║
    ║      → stress → +1.0 → FREEZE + GROW                            ║
    ║                                                                  ║
    ║    After freeze+grow:                                            ║
    ║      Old connections: frozen, gradient = 0 → stress decays to 0 ║
    ║      New neurons: learn XOR freely, consistent direction         ║
    ║      → conflict falls → network stable again                    ║
    ╚══════════════════════════════════════════════════════════════════╝

    Replay is used for JOINT TRAINING only (mixing past data into the
    current loss) — NOT for forgetting detection.  The stress signal
    is what creates the conflict: Circles replay data creates opposing
    gradients when XOR is being learned, naturally raising stress on
    the shared weights.
    """
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.MSELoss()

    net.to(device)
    X, y = X.to(device), y.to(device)

    loss_ema = 0.25

    # ── Stress monitoring ─────────────────────────────────────────────────────
    # stress_ema: smoothed network-level conflict fraction (in [0, 1])
    # Starts at 0 (no conflict assumed at start of each call).
    stress_ema          = 0.0
    stress_ema_beta     = 0.9       # smooth over ~10 checks = ~100 batches
    stress_check_every  = 10        # batches between stress evaluations

    # Conflict threshold: importance-weighted conflict fraction above which
    # the network is considered to be "under stress from a new task."
    #
    # Task 1 alone (no replay):   conflict ≈ 0.00–0.15  (consistent gradients)
    # Task 2 (Circles vs XOR):    conflict → 0.40–0.90  (opposing gradients)
    #
    # Threshold at 0.3 sits well between these two regimes.
    # Increase if too many false freezes during Task 1.
    # Decrease if freeze fires too late into Task 2.
    conflict_threshold  = 0.3

    # Burn-in: don't allow any freeze in the first N batches.
    # Gives the network time to start learning (gradients are chaotic
    # at initialization regardless of task conflict).
    freeze_burnin       = 300       # ≈ 30 epochs with batch_size=32, n=300

    # Cooldown: minimum batches between successive freeze events.
    # Prevents rapid-fire freezing before new neurons have had time
    # to relieve the stress.
    freeze_cooldown     = 500       # ≈ 50 epochs
    batches_since_freeze = freeze_cooldown   # start ready
    neurons_per_freeze   = 3

    # ── Task-1 conservative growth ────────────────────────────────────────────
    # During Task 1 there is no replay conflict, so stress stays low and the
    # conflict-based freeze never fires.  Instead, we grow when the task loss
    # is still high (network genuinely doesn't have enough capacity).
    task1_grow_cooldown   = 200
    batches_since_t1_grow = task1_grow_cooldown
    task1_loss_threshold  = 0.15
    task1_max_neurons     = 20      # hard cap: prevents runaway growth on easy tasks

    total_batches = 0

    for epoch in range(epochs):
        net.train()
        perm = torch.randperm(X.size(0))

        for i in range(0, X.size(0), 32):
            idx     = perm[i:i + 32]
            batch_x = X[idx]
            batch_y = y[idx]

            optimizer.zero_grad()

            # ── 1. Task Gradient ──────────────────────────────────────────────
            optimizer.zero_grad()
            task_loss = criterion(net(batch_x), batch_y)
            task_loss.backward(retain_graph=True)
            g_task = net.W.grad.clone() if net.W.grad is not None else torch.zeros_like(net.W)

            # ── 2. Replay Gradient (creates the conflict signal) ──────────────
            optimizer.zero_grad()
            replay_val = torch.tensor(0.0, device=device)
            g_replay = torch.zeros_like(net.W)
            if replay is not None and len(replay) >= 16:
                rx, ry = replay.sample(16, device)
                if rx is not None:
                    replay_val = criterion(net(rx), ry)
                    replay_val.backward(retain_graph=True)
                    g_replay = net.W.grad.clone() if net.W.grad is not None else torch.zeros_like(net.W)

            # ── 3. Update Stress Signal ───────────────────────────────────────
            if replay is not None and len(replay) >= 16:
                net.update_stress(g_task, g_replay)

            # ── 4. Actual Optimizer Step (Combined) ───────────────────────────
            optimizer.zero_grad()
            combined_loss = task_loss + replay_val
            combined_loss.backward()
            optimizer.step()

            # Push batch to replay (AFTER step, so this batch's data is
            # available for future tasks — not this batch's training)
            if replay is not None:
                replay.push(batch_x, batch_y)

            loss_ema = 0.99 * loss_ema + 0.01 * task_loss.item()
            batches_since_freeze  += 1
            batches_since_t1_grow += 1

            # ── Stress check ──────────────────────────────────────────────────
            if total_batches % stress_check_every == 0:
                s = net.get_network_stress()
                stress_ema = stress_ema_beta * stress_ema + (1 - stress_ema_beta) * s

                # ── Conflict-triggered FREEZE + GROW ─────────────────────────
                if (total_batches >= freeze_burnin and
                        batches_since_freeze >= freeze_cooldown and
                        stress_ema > conflict_threshold):

                    n_frozen = net.stress_freeze(threshold=0.3)

                    if n_frozen > 0:
                        for _ in range(neurons_per_freeze):
                            net.grow_neuron(num_connections=3)

                        # Reset stress buffers for new neurons
                        with torch.no_grad():
                            new_trainable = net.trainable_mask * net.M
                            net.stress_num *= (1.0 - new_trainable)
                            net.stress_den *= (1.0 - new_trainable)

                        # Reset Adam state for new neurons
                        for ni in range(net.active_neurons - neurons_per_freeze,
                                        net.active_neurons):
                            _reset_adam_for_neuron(optimizer, net, ni)

                        batches_since_freeze = 0
                        # Snap stress_ema down: after freeze, conflict should
                        # reduce — don't let stale high value trigger another
                        # freeze immediately after cooldown expires.
                        stress_ema = 0.0

                        print(f"  [Epoch {epoch+1:4d}] STRESS-FREEZE: "
                              f"{n_frozen} conns frozen, grew {neurons_per_freeze} → "
                              f"active={net.active_neurons} "
                              f"(trainable={net.n_trainable()}, frozen={net.n_frozen()}) "
                              f"conflict={s:.3f}")

                # ── Loss-based growth (when there is no conflict) ─────────────
                # If loss is high, but stress is low, the network doesn't have 
                # enough capacity for the new task, but the new task isn't 
                # fighting the old ones (orthogonal). So just grow!
                elif (net.active_neurons < task1_max_neurons and
                      loss_ema > task1_loss_threshold and
                      batches_since_t1_grow >= task1_grow_cooldown and
                      stress_ema <= conflict_threshold):

                    net.grow_neuron(num_connections=3)
                    ni = net.active_neurons - 1
                    _reset_adam_for_neuron(optimizer, net, ni)
                    batches_since_t1_grow = 0
                    print(f"  [Epoch {epoch+1:4d}] GROW(Loss): "
                          f"active={net.active_neurons} loss={loss_ema:.4f}")

            total_batches += 1

    return net


# ─────────────────────────────────────────────────────────────────────────────
def evaluate(net, tasks, seen_up_to, device):
    net.eval()
    with torch.no_grad():
        for name, X_np, y_np in tasks[:seen_up_to + 1]:
            X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
            y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
            acc = ((net(X_t) > 0.5).float() == y_t).float().mean().item()
            print(f"  Accuracy on {name}: {acc * 100:.1f}%")


# ─────────────────────────────────────────────────────────────────────────────



In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

base_tasks = [
    ("Circles", *make_circles(n_samples=300, noise=0.05, seed=42, center=(0, 0))),
    ("XOR",     *make_xor(n_samples=300, noise=0.0,  seed=42, center=(4, 4))),
    ("Linear",  *make_linearly_separable(n_samples=300, noise=0.1, seed=42, center=(-4, -4))),
]

class BaselineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 14),
            nn.Sigmoid(),
            nn.Linear(14, 1)
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x))

def run_experiment(task_order):
    perm_names = [t[0] for t in task_order]
    print(f"\n{'='*60}\nRunning Permutation: {perm_names}\n{'='*60}")
    
    print("\n--- Training Baseline ---")
    baseline_net = BaselineNet().to(device)
    optimizer_b = torch.optim.Adam(baseline_net.parameters(), lr=3e-3)
    criterion_b = nn.BCELoss()
    
    for task_id, (name, X_np, y_np) in enumerate(task_order):
        X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
        y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
        baseline_net.train()
        for epoch in range(1000):
            optimizer_b.zero_grad()
            loss = criterion_b(baseline_net(X_t), y_t)
            loss.backward()
            optimizer_b.step()
            
    baseline_accs = []
    baseline_net.eval()
    with torch.no_grad():
        for name, X_np, y_np in task_order:
            X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
            y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
            acc = ((baseline_net(X_t) > 0.5).float() == y_t).float().mean().item()
            baseline_accs.append((name, acc))
            
    print("\n--- Training Dynamic Network ---")
    dynamic_net = PyTorchDynamicNetwork(input_dim=2, output_dim=1, max_neurons=200).to(device)
    replay = ReplayBuffer(capacity=500)
    
    for task_id, (name, X_np, y_np) in enumerate(task_order):
        print(f"  Training {name}...")
        X_t = torch.tensor(X_np, dtype=torch.float32)
        y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1)
        train_one_task_gpu(dynamic_net, X_t, y_t, epochs=1000, lr=3e-3, replay=replay, device=device)
        
    dynamic_accs = []
    dynamic_net.eval()
    with torch.no_grad():
        for name, X_np, y_np in task_order:
            X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
            y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
            acc = ((dynamic_net(X_t) > 0.5).float() == y_t).float().mean().item()
            dynamic_accs.append((name, acc))
            
    return baseline_net, dynamic_net, baseline_accs, dynamic_accs

results = []
permutations = list(itertools.permutations(base_tasks))

for perm in permutations:
    b_net, d_net, b_accs, d_accs = run_experiment(perm)
    results.append({
        'perm': [t[0] for t in perm],
        'b_net': b_net,
        'd_net': d_net,
        'b_accs': b_accs,
        'd_accs': d_accs
    })

print("\n" + "="*80)
print("FINAL SUMMARY (All 6 Permutations)")
print("="*80)
for r in results:
    perm_str = " -> ".join(r['perm'])
    print(f"Order: {perm_str:25s}")
    b_str = ", ".join([f"{n}: {a*100:.1f}%" for n, a in r['b_accs']])
    d_str = ", ".join([f"{n}: {a*100:.1f}%" for n, a in r['d_accs']])
    print(f"  Baseline Acc : {b_str}")
    print(f"  Dynamic Acc  : {d_str}\n")



In [ ]:

def plot_dynamic_topology(net, title):
    G = nx.DiGraph()
    n = net.active_neurons
    M = net.M[:n, :n].cpu().numpy()
    trainable = net.trainable_mask[:n, :n].cpu().numpy()
    
    colors = []
    for i in range(n):
        G.add_node(i)
        if i < net.input_dim:
            colors.append('lightblue')
        elif i < net.input_dim + net.output_dim:
            colors.append('lightcoral')
        else:
            colors.append('lightgray')
            
    edge_colors = []
    for i in range(n):
        for j in range(n):
            if M[i, j] == 1.0:
                G.add_edge(i, j)
                if trainable[i, j] == 1.0:
                    edge_colors.append('green')
                else:
                    edge_colors.append('red')
                    
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, node_color=colors, edge_color=edge_colors, with_labels=True, 
            node_size=600, arrows=True, font_size=10, font_weight='bold', 
            connectionstyle='arc3,rad=0.1')
    
    import matplotlib.patches as mpatches
    g_patch = mpatches.Patch(color='green', label='Trainable Connection')
    r_patch = mpatches.Patch(color='red', label='Frozen Connection')
    in_patch = mpatches.Patch(color='lightblue', label='Input Neuron')
    hid_patch = mpatches.Patch(color='lightgray', label='Hidden Neuron')
    out_patch = mpatches.Patch(color='lightcoral', label='Output Neuron')
    
    plt.legend(handles=[g_patch, r_patch, in_patch, hid_patch, out_patch], loc='upper left')
    plt.title(title, fontsize=14)
    plt.show()

last_d_net = results[-1]['d_net']
last_perm_str = " -> ".join(results[-1]['perm'])
plot_dynamic_topology(last_d_net, f"Dynamic Network Topology (After sequence: {last_perm_str})")



In [ ]:

b_net = results[-1]['b_net']
d_net = results[-1]['d_net']

def plot_model(model_net, ax, title):
    model_net.eval()
    x_min, x_max = -8, 8
    y_min, y_max = -8, 8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_tensor = torch.tensor(grid, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        Z = model_net(grid_tensor).cpu().numpy()
        Z = (Z > 0.5).astype(int).reshape(xx.shape)
        
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    
    colors = ['blue', 'red', 'green']
    markers = ['o', 's', '^']
    
    for i, (name, X_np, y_np) in enumerate(base_tasks):
        c0 = X_np[y_np == 0]
        c1 = X_np[y_np == 1]
        
        ax.scatter(c0[:, 0], c0[:, 1], c=colors[i], marker=markers[0], edgecolor='k', label=f'{name} (Class 0)' if i==0 else "", alpha=0.6)
        ax.scatter(c1[:, 0], c1[:, 1], c=colors[i], marker=markers[1], edgecolor='k', label=f'{name} (Class 1)' if i==0 else "", alpha=0.6)
        
    ax.set_title(title, fontsize=14)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

plot_model(b_net, ax1, f"Baseline Network\n(Catastrophic Forgetting after {last_perm_str})")
plot_model(d_net, ax2, f"Our Dynamic Network\n(Zero Forgetting after {last_perm_str}!)")

plt.tight_layout()
plt.show()

